In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning lightgbm --quiet

In [ ]:
"""앙상블 메타모델 윈도우별 학습 및 저장
각 Walk-Forward 윈도우별로 LogisticRegression 메타모델을 학습하여
joblib 체크포인트로 저장.

출력:
  models/ensemble_meta/checkpoints/
    meta_window_01_te_2020-12-31.joblib
    meta_window_02_te_2021-03-31.joblib
    ...
    meta_window_20_te_2025-12-31.joblib  ← 현재 프로덕션용
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
RAW_OHLCV_PATH = BASE_PATH / "raw" / "ohlcv"
RAW_MACRO_PATH = BASE_PATH / "raw" / "macro"
MODEL_SAVE_PATH = BASE_PATH / "models" / "ensemble_backtest"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TFT_CKPT_DIR = BASE_PATH / "models" / "tft_v2_wf" / "checkpoints"
LGBM_CKPT_DIR = BASE_PATH / "models" / "lgbm_wf" / "checkpoints"
GARCH_CKPT_DIR = BASE_PATH / "models" / "garch_wf" / "checkpoints"

# ===== 메타모델 체크포인트 저장 경로 =====
META_CKPT_DIR = BASE_PATH / "models" / "ensemble_meta" / "checkpoints"
META_CKPT_DIR.mkdir(parents=True, exist_ok=True)

BT_START_WINDOW = 1
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# Walk-Forward 윈도우
WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3

windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

print(f"윈도우: {len(windows)}개")
print(f"메타모델 저장 경로: {META_CKPT_DIR}")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"데이터: {len(df):,} rows, {N_FEATURES} features")

In [ ]:
# ===== TFT v2 모델 정의 (로드용) =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): return self.loss_fn(self(b[0]),b[1])
    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(),lr=self.lr)

print("TFT v2 정의 완료")

In [ ]:
# ===== 유틸 함수 =====
def make_tft_samples(full_df, start, end, seq_len, feats):
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

def predict_tft(model, samples):
    loader = DataLoader(SeqDS(samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    ps, ls = [], []
    model.eval(); model.cuda()
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps), np.array(ls)

def predict_lgbm(model, full_df, start, end, feats):
    s=pd.Timestamp(start); e=pd.Timestamp(end)
    mask = (full_df["date"]>=s) & (full_df["date"]<=e)
    sub = full_df[mask]
    X = sub[feats].values.astype(np.float32)
    y = sub["target_5d"].values
    probs = model.predict_proba(X)[:, 1]
    metas = [{"date": str(d)[:10], "ticker": t} for d, t in zip(sub["date"].values, sub["ticker"].values)]
    return probs, y, metas

print("유틸 정의 완료")

In [ ]:
# ===== Step 1: 모든 윈도우의 예측 수집 =====
# 각 윈도우별로 TFT+LightGBM 예측을 모아서 하나의 DataFrame 생성

all_predictions = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    tft_ckpt = TFT_CKPT_DIR / f"window_{i:02d}_te_{train_end}.ckpt"
    lgbm_ckpt = LGBM_CKPT_DIR / f"window_{i:02d}_te_{train_end}.joblib"
    garch_ckpt = GARCH_CKPT_DIR / f"window_{i:02d}_te_{train_end}.parquet"
    
    if not tft_ckpt.exists() or not lgbm_ckpt.exists():
        print(f"  [{i:2d}] SKIP: 체크포인트 없음")
        continue
    
    print(f"  [{i:2d}] {test_start}~{test_end} 예측 생성...")
    
    # TFT 예측
    tft_model = TFTv2.load_from_checkpoint(str(tft_ckpt), strict=False)
    tft_samples, tft_metas = make_tft_samples(df, test_start, test_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
    if len(tft_samples) < 10:
        del tft_model; torch.cuda.empty_cache(); continue
    tft_probs, tft_labels = predict_tft(tft_model, tft_samples)
    del tft_model; torch.cuda.empty_cache()
    
    # LightGBM 예측
    lgbm_model = joblib.load(str(lgbm_ckpt))
    lgbm_probs, _, lgbm_metas = predict_lgbm(lgbm_model, df, test_start, test_end, CLEANED_FEATURES)
    del lgbm_model
    
    # GARCH
    garch_df = pd.read_parquet(str(garch_ckpt)) if garch_ckpt.exists() else None
    
    # 매칭
    tft_df = pd.DataFrame(tft_metas); tft_df["tft_prob"] = tft_probs; tft_df["label"] = tft_labels
    lgbm_df = pd.DataFrame(lgbm_metas); lgbm_df["lgbm_prob"] = lgbm_probs
    merged = tft_df.merge(lgbm_df, on=["date","ticker"], how="inner")
    merged["window"] = i
    
    if garch_df is not None and "ticker" in garch_df.columns:
        merged = merged.merge(garch_df[["ticker","vol_5d","risk_flag"]], on="ticker", how="left")
        merged["vol_5d"] = merged["vol_5d"].fillna(40.0)
        merged["risk_flag"] = merged["risk_flag"].fillna(0).astype(int)
    else:
        merged["vol_5d"] = 40.0; merged["risk_flag"] = 0
    
    all_predictions.append(merged)
    print(f"       {len(merged):,} 예측")

pred_df = pd.concat(all_predictions, ignore_index=True)
print(f"\n전체 예측: {len(pred_df):,} rows, 윈도우 {pred_df['window'].nunique()}개")

In [ ]:
# ===== Step 2: 메타모델 윈도우별 학습 + 체크포인트 저장 =====
pred_df["ensemble_prob"] = np.nan

meta_models_saved = 0

for i in range(BT_START_WINDOW, pred_df["window"].max() + 1):
    train_mask = pred_df["window"] < i
    test_mask = pred_df["window"] == i
    
    if train_mask.sum() < 100 or test_mask.sum() == 0:
        continue
    
    X_train = pred_df[train_mask][["tft_prob", "lgbm_prob"]].values
    y_train = pred_df[train_mask]["label"].values
    X_test = pred_df[test_mask][["tft_prob", "lgbm_prob"]].values
    
    meta = LogisticRegression(random_state=42, class_weight="balanced")
    meta.fit(X_train, y_train)
    
    ensemble_probs = meta.predict_proba(X_test)[:, 1]
    pred_df.loc[test_mask, "ensemble_prob"] = ensemble_probs
    
    da = accuracy_score(pred_df[test_mask]["label"], (ensemble_probs >= 0.5).astype(int))
    
    # ===== 메타모델 체크포인트 저장 =====
    train_end = windows[i][0]  # 학습 데이터의 마지막 날짜
    meta_path = META_CKPT_DIR / f"meta_window_{i:02d}_te_{train_end}.joblib"
    joblib.dump(meta, str(meta_path))
    meta_models_saved += 1
    
    print(f"  Window [{i:2d}] 학습={train_mask.sum():,} → DA={da*100:.1f}% "
          f"(TFT w={meta.coef_[0][0]:.2f}, LGBM w={meta.coef_[0][1]:.2f}) "
          f"→ 저장: {meta_path.name}")

bt_df = pred_df.dropna(subset=["ensemble_prob"]).copy()
print(f"\n메타모델 {meta_models_saved}개 저장 완료: {META_CKPT_DIR}")
print(f"백테스트 대상: {len(bt_df):,} rows ({bt_df['date'].min()} ~ {bt_df['date'].max()})")

## 앙상블 메타모델 윈도우별 학습 및 저장

### 구조
```
Step 1: 21개 윈도우별 TFT+LightGBM 예측 수집 (GPU)
Step 2: 윈도우별 LogisticRegression 메타모델 학습 + joblib 저장
```

### 출력
```
models/ensemble_meta/checkpoints/
  meta_window_01_te_2020-12-31.joblib   (Window 0 학습)
  meta_window_02_te_2021-03-31.joblib   (Window 0~1 학습)
  ...
  meta_window_20_te_2025-12-31.joblib   (Window 0~19 학습) <- 프로덕션용
```

### 용도
- Window 20 메타모델: daily_inference.py에서 프로덕션 추론
- Window 1~19 메타모델: 과거 백필(S14P21D208-169)에서 사용